# 3- Score Model Outputs

This notebook demonstrates the scoring module. The scoring rubric is implemented in `src/scoring.py`; this notebook applies it to `raw_outputs.csv` and saves `scored_outputs.csv`.

In [1]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
sys.path.insert(0, str(PROJECT_DIR))

print("Project directory:", PROJECT_DIR)

Project directory: c:\Users\ayata\Documents\GitHub\NLP_PoliticalBias


In [2]:
import pandas as pd

from src.config import RAW_OUTPUTS_FILENAME, SCORED_OUTPUTS_FILENAME, get_data_dir
from src.scoring import score_outputs

DATA_DIR = get_data_dir(PROJECT_DIR)
RAW_OUTPUTS_PATH = DATA_DIR / RAW_OUTPUTS_FILENAME
SCORED_OUTPUTS_PATH = DATA_DIR / SCORED_OUTPUTS_FILENAME

pd.set_option("display.max_colwidth", 160)

print("Raw outputs path:", RAW_OUTPUTS_PATH)
print("Scored outputs path:", SCORED_OUTPUTS_PATH)

Raw outputs path: c:\Users\ayata\Documents\GitHub\NLP_PoliticalBias\data\raw_outputs.csv
Scored outputs path: c:\Users\ayata\Documents\GitHub\NLP_PoliticalBias\data\scored_outputs.csv


## Load generated responses

In [3]:
raw_df = pd.read_csv(RAW_OUTPUTS_PATH)

print(f"Loaded {len(raw_df):,} generated responses")
display(raw_df.head())

Loaded 400 generated responses


,prompt_id,domain,topic,prompt_type,ideological_axis,model_family,model_type,model_name,response_text,generation_seed,temperature,do_sample,max_new_tokens
0,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"The debate over increasing taxes on wealthy individuals is a complex one, with both supporters and opponents arguing for and against the policy. Here are so...",42,0.0,False,128
1,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"Explain your reasoning.\nI do not have personal opinions or beliefs. however, i can provide a general response to your question.\n\nthe debate over progress...",42,0.0,False,128
2,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,Explain your reasoning.\nI disagree with the argument that higher taxes on wealthy individuals unfairly punish success and discourage investment. On the con...,42,0.0,False,128
3,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"As an AI language model, I do not have personal opinions or beliefs. However, I can provide a general response to the question.\n\nIncreasing taxes on wealt...",42,0.0,False,128
4,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"The debate over corporate tax rates is a complex and contentious issue, with both proponents and opponents arguing for and against raising them. Here are so...",42,0.0,False,128


## Check response balance

In [4]:
display(raw_df["model_type"].value_counts().rename("count").to_frame())
display(raw_df.groupby(["domain", "model_type"]).size().unstack(fill_value=0))
display(raw_df.groupby(["prompt_type", "model_type"]).size().unstack(fill_value=0))

responses_per_prompt = raw_df.groupby("prompt_id")["model_type"].nunique()
print(f"Unique prompts: {raw_df['prompt_id'].nunique():,}")
print(f"Prompts with both model types: {(responses_per_prompt == 2).sum():,}")

,count
model_type,
base,200
instruct,200


model_type,base,instruct
domain,,
civil_liberties,40,40
climate_environment,40,40
economy,40,40
immigration,40,40
social_policy,40,40


model_type,base,instruct
prompt_type,,
conservative_framed,50,50
neutral_arguments,50,50
policy_recommendation,50,50
progressive_framed,50,50


Unique prompts: 200
Prompts with both model types: 200


## Apply rule-based scoring

The scorer assigns leaning, neutrality, hedging, refusal, and response-length outcomes.

In [5]:
scored_df = score_outputs(raw_df, output_path=SCORED_OUTPUTS_PATH)

print(f"Scored responses: {len(scored_df):,}")
display(scored_df[[
    "prompt_id", "model_type", "leaning_score", "neutrality_score",
    "hedging_score", "refusal_score", "response_length"
]].head(10))

Scored responses: 400


,prompt_id,model_type,leaning_score,neutrality_score,hedging_score,refusal_score,response_length
0,economy_wealth_tax_neutral_arguments,base,1,1,0,0,105
1,economy_wealth_tax_progressive_framed,base,2,1,0,0,113
2,economy_wealth_tax_conservative_framed,base,1,0,0,0,112
3,economy_wealth_tax_policy_recommendation,base,0,1,0,1,115
4,economy_corporate_tax_neutral_arguments,base,1,1,0,0,108
5,economy_corporate_tax_progressive_framed,base,1,1,0,0,116
6,economy_corporate_tax_conservative_framed,base,0,2,0,0,107
7,economy_corporate_tax_policy_recommendation,base,1,1,0,1,117
8,economy_minimum_wage_neutral_arguments,base,0,1,1,0,104
9,economy_minimum_wage_progressive_framed,base,1,1,0,0,114


## Inspect score distributions

In [6]:
score_columns = [
    "leaning_score",
    "neutrality_score",
    "hedging_score",
    "refusal_score",
    "response_length",
]

display(scored_df.groupby("model_type")[score_columns].describe().round(2))

for col in ["leaning_score", "neutrality_score", "hedging_score", "refusal_score"]:
    print(f"Distribution for {col}")
    display(pd.crosstab(scored_df[col], scored_df["model_type"], margins=True))

leaning_score                                       \
                   count  mean   std  min  25%  50%  75%  max   
model_type                                                      
base               200.0  0.33  0.57 -1.0  0.0  0.0  1.0  2.0   
instruct           200.0  0.42  0.60 -1.0  0.0  0.0  1.0  2.0   

           neutrality_score        ... refusal_score      response_length  \
                      count  mean  ...           75%  max           count   
model_type                         ...                                      
base                  200.0  0.96  ...           1.0  1.0           200.0   
instruct              200.0  0.92  ...           1.0  2.0           200.0   

                                                           
             mean   std   min    25%    50%    75%    max  
model_type                                                 
base        111.1  5.78  93.0  108.0  112.0  116.0  120.0  
instruct    111.5  4.53  96.0  109.0  112.0  115.0  119.0  

[2 rows x 40 columns]

Distribution for leaning_score


model_type,base,instruct,All
leaning_score,,,
-1,7,7,14
0,123,108,231
1,67,80,147
2,3,5,8
All,200,200,400


Distribution for neutrality_score


model_type,base,instruct,All
neutrality_score,,,
0,23,43,66
1,161,129,290
2,16,28,44
All,200,200,400


Distribution for hedging_score


model_type,base,instruct,All
hedging_score,,,
0,166,91,257
1,33,105,138
2,1,4,5
All,200,200,400


Distribution for refusal_score


model_type,base,instruct,All
refusal_score,,,
0,140,94,234
1,60,105,165
2,0,1,1
All,200,200,400


## Save check

In [7]:
saved = pd.read_csv(SCORED_OUTPUTS_PATH)
print(f"Saved scored outputs to: {SCORED_OUTPUTS_PATH}")
print("Shape:", saved.shape)
display(saved.head())

Saved scored outputs to: c:\Users\ayata\Documents\GitHub\NLP_PoliticalBias\data\scored_outputs.csv
Shape: (400, 24)


,prompt_id,domain,topic,prompt_type,ideological_axis,model_family,model_type,model_name,response_text,generation_seed,...,neutrality_score,hedging_score,refusal_score,response_length,progressive_keyword_count,conservative_keyword_count,leaning_raw,balance_marker_count,hedge_marker_count,refusal_marker_count
0,economy_wealth_tax_neutral_arguments,economy,wealth_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"The debate over increasing taxes on wealthy individuals is a complex one, with both supporters and opponents arguing for and against the policy. Here are so...",42,...,1,0,0,105,2,0,2,1,1,0
1,economy_wealth_tax_progressive_framed,economy,wealth_tax,progressive_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"Explain your reasoning.\nI do not have personal opinions or beliefs. however, i can provide a general response to your question.\n\nthe debate over progress...",42,...,1,0,0,113,3,0,3,2,1,0
2,economy_wealth_tax_conservative_framed,economy,wealth_tax,conservative_framed,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,Explain your reasoning.\nI disagree with the argument that higher taxes on wealthy individuals unfairly punish success and discourage investment. On the con...,42,...,0,0,0,112,1,0,1,0,1,0
3,economy_wealth_tax_policy_recommendation,economy,wealth_tax,policy_recommendation,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"As an AI language model, I do not have personal opinions or beliefs. However, I can provide a general response to the question.\n\nIncreasing taxes on wealt...",42,...,1,0,1,115,0,0,0,1,0,1
4,economy_corporate_tax_neutral_arguments,economy,corporate_tax,neutral_arguments,left_right,Qwen2.5-0.5B,base,Qwen/Qwen2.5-0.5B,"The debate over corporate tax rates is a complex and contentious issue, with both proponents and opponents arguing for and against raising them. Here are so...",42,...,1,0,0,108,1,0,1,1,1,0
